In [ ]:
import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
def load_and_clean_transactions(file_path: str = None) -> pd.DataFrame:
    """Loads transactions and filters out returns, missing users, and invalid records."""
    if file_path:
        df = pd.read_csv(file_path)
    else:
        # Synthetic fallback data
        np.random.seed(42)
        n = 5000
        customers = [f"CUST-{i:03d}" for i in range(1, 200)] + [np.nan] * 20
        df = pd.DataFrame({
            "InvoiceNo": [f"INV-{1000 + i}" for i in range(n)],
            "CustomerID": np.random.choice(customers, size=n),
            "InvoiceDate": pd.date_range("2025-01-01", "2026-08-01", periods=n),
            "Quantity": np.random.choice([1, 2, 5, -1], size=n, p=[0.7, 0.2, 0.05, 0.05]),
            "UnitPrice": np.random.uniform(5, 100, size=n)
        })

    # Drop missing customer records & valid sales filter
    clean_df = df.dropna(subset=["CustomerID"]).copy()
    clean_df = clean_df[(clean_df["Quantity"] > 0) & (clean_df["UnitPrice"] > 0)]
    clean_df["LineTotal"] = clean_df["Quantity"] * clean_df["UnitPrice"]
    
    return clean_df

In [ ]:
def compute_rfm(df: pd.DataFrame) -> pd.DataFrame:
    """Calculates Recency, Frequency, and Monetary scores per customer."""
    ref_date = df["InvoiceDate"].max() + dt.timedelta(days=1)
    
    rfm = df.groupby("CustomerID").agg({
        "InvoiceDate": lambda dates: (ref_date - dates.max()).days,
        "InvoiceNo": "nunique",
        "LineTotal": "sum"
    }).reset_index()

    rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]
    return rfm


In [ ]:
def transform_features(rfm_df: pd.DataFrame):
    """Log-transforms heavy-skew features and applies standard scaling."""
    rfm_log = np.log1p(rfm_df[["Recency", "Frequency", "Monetary"]])
    scaler = StandardScaler()
    scaled_matrix = scaler.fit_transform(rfm_log)
    return scaled_matrix, scaler

In [ ]:
def run_rfm_clustering():
    df = load_and_clean_transactions()
    rfm = compute_rfm(df)
    scaled_data, _ = transform_features(rfm)

    print(" Evaluating cluster range (K=2 to 6)...")
    for k in range(2, 7):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(scaled_data)
        sil = silhouette_score(scaled_data, labels)
        print(f"  • K={k} | Silhouette Score: {sil:.4f}")

    # Select K=4 for RFM segmentation
    optimal_k = 4
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    rfm["Cluster"] = kmeans.fit_predict(scaled_data)

    # Calculate cluster summaries
    profile = rfm.groupby("Cluster").agg({
        "Recency": "mean",
        "Frequency": "mean",
        "Monetary": "mean",
        "CustomerID": "count"
    }).rename(columns={"CustomerID": "CustomerCount"}).round(2)

    # Rank and map human-readable business segment names
    profile["Score"] = profile["Monetary"] + profile["Frequency"] - profile["Recency"]
    sorted_clusters = profile.sort_values(by="Score", ascending=False).index

    labels = ["Champions", "Loyal Customers", "At-Risk", "Hibernating"]
    segment_map = {cluster_id: label for cluster_id, label in zip(sorted_clusters, labels)}
    
    rfm["Segment"] = rfm["Cluster"].map(segment_map)

    print("\n Segment Profiles:")
    final_summary = rfm.groupby("Segment").agg({
        "Recency": "mean",
        "Frequency": "mean",
        "Monetary": "mean",
        "CustomerID": "count"
    }).rename(columns={"CustomerID": "Count"}).round(1)
    
    print(final_summary)

    # Simple scatter visualization
    sns.scatterplot(data=rfm, x="Recency", y="Monetary", hue="Segment", palette="viridis")
    plt.title("Customer Segments: Recency vs Monetary")
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    run_rfm_clustering()